# Dask quickstart — `NetCDF`

Read one variable lazily, across time and across files. `read_array(variable, chunks=...)`
returns a `dask.array`; `open_mfdataset` stacks many files into one. Kerchunk manifests
(zero-copy byte-range indexes) need the `[lazy]` extra. For the full surface see
`lazy-netcdf-complete`.

## Setup

In [1]:
import os

os.environ['MPLBACKEND'] = 'Agg'  # never trigger an interactive backend

from pathlib import Path

import dask.array as da
import numpy as np


DATA = Path('../../../examples/data')
if not DATA.exists():
    DATA = Path('examples/data')
DATA.is_dir()

True

## Eager vs lazy variable reads

In [2]:
from pyramids.netcdf import NetCDF

nc = NetCDF.read_file(str(DATA / 'netcdf' / 'pyramids-netcdf-3d.nc'))
nc.variable_names

2026-06-07 21:12:08 | INFO | pyramids.base.config | Logging is configured.


['values']

In [3]:
eager = nc.read_array('values')  # numpy
lazy = nc.read_array('values', chunks=(1, -1, -1))  # dask — one chunk per leading step
(type(eager).__name__, eager.shape), (type(lazy).__name__, lazy.chunks)

(('ndarray', (3, 13, 14)), ('Array', ((1, 1, 1), (13,), (14,))))

## Reduce over the leading axis

In [4]:
# No I/O until compute().
per_cell_mean = lazy.mean(axis=0)
per_cell_mean.compute().shape

C:\gdrive\algorithms\gis\pyramids\.claude\worktrees\dask-notebooks\.pixi\envs\dev\Lib\site-packages\numpy\_core\_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


(13, 14)

## Multi-file stacks — `open_mfdataset`

One variable at a time; returns a `(n_files, *var_shape)` dask array. Here we reuse one
file three times to stand in for a real multi-file cube.

In [5]:
paths = [str(DATA / 'netcdf' / 'pyramids-netcdf-3d.nc')] * 3
stack = NetCDF.open_mfdataset(paths, variable='values')
stack.shape, stack.mean(axis=0).compute().shape

C:\gdrive\algorithms\gis\pyramids\.claude\worktrees\dask-notebooks\.pixi\envs\dev\Lib\site-packages\numpy\_core\_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


((3, 3, 13, 14), (3, 13, 14))

## Kerchunk manifests (optional)

`to_kerchunk` writes a JSON byte-range index so the file can be opened zero-copy by
downstream tools. It needs the `[lazy]` extra — guard the call so the notebook
still runs without it.

In [ ]:
import tempfile

manifest = Path(tempfile.mkdtemp(prefix='pyramids-nc-')) / 'single.json'
try:
    refs = nc.to_kerchunk(str(manifest))
    outcome = 'wrote manifest:', manifest.exists()
except ImportError as exc:
    outcome = 'lazy extra missing:', str(exc)
outcome

## Where to next

Three tiers for `NetCDF` + Dask:

- **Complete cookbook (offline)** —
  [Lazy `NetCDF`](../netcdf/lazy-netcdf-complete.ipynb): `unpack`, combined kerchunk
  manifests, handing arrays to xarray.
- **Cloud (live data)** —
  [Dask on ERA5 (AWS)](../netcdf/dask-lazy-netcdf.ipynb): kerchunk over a decade of
  reanalysis.
- **Narrative guide** — the [Lazy NetCDF tutorial](../../tutorials/lazy/lazy-netcdf.md).
- **All classes at a glance** — [the overview](overview.ipynb).